# Fraud Detection ML Pipeline

End-to-end notebook for training, registering, deploying, and configuring shadow/canary traffic splitting for fraud detection models on Snowflake.

**Pipeline Steps:**
1. Connect to Snowflake using SPCS container token
2. Load fraud dataset from a local (workspace) CSV file
3. Feature engineering
4. Train champion (XGBoost) and challenger (Random Forest) models
5. Register both models in the Snowflake Model Registry
6. Create compute pool and deploy model services
7. Create a gateway with traffic splitting for shadow/canary deployment

**Prerequisites:**
- Running inside a Snowpark Container Services environment
- `fraud_demo_dataset.csv` included in the notebook workspace
- Role with CREATE MODEL, CREATE SERVICE, CREATE GATEWAY, BIND SERVICE ENDPOINT privileges

**Required Grants:**
```sql
GRANT CREATE MODEL ON SCHEMA ERIC_DB.PUBLIC TO ROLE SPCS_DEMO_PROVIDER_ROLE;
GRANT CREATE SERVICE ON SCHEMA ERIC_DB.PUBLIC TO ROLE SPCS_DEMO_PROVIDER_ROLE;
GRANT CREATE GATEWAY ON SCHEMA ERIC_DB.PUBLIC TO ROLE SPCS_DEMO_PROVIDER_ROLE;
GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE SPCS_DEMO_PROVIDER_ROLE;
GRANT USAGE ON COMPUTE POOL SYSTEM_COMPUTE_POOL_CPU TO ROLE SPCS_DEMO_PROVIDER_ROLE;
```

In [ ]:
# Snowflake connection settings.
# Database, schema, and role used for all objects created by this notebook.
SF_WAREHOUSE = "<warehouse>"
SF_DATABASE = "<database>"
SF_SCHEMA = "<schema>"
SF_ROLE = "<role>"

# Data source configuration.
# Path to the fraud dataset CSV file inside the SPCS container.
LOCAL_DATA_PATH = "fraud_demo_dataset.csv"

# Model registry names and version.
# Champion (XGBoost) and challenger (Random Forest) identifiers in the registry.
CHAMPION_MODEL_NAME = "FRAUD_DETECTION_XGBOOST"
CHALLENGER_MODEL_NAME = "FRAUD_DETECTION_RF"
MODEL_VERSION = "V1"

# SPCS service names for the deployed model endpoints.
# These are the service objects created by model_version.create_service().
CHAMPION_SERVICE_NAME = "FRAUD_XGBOOST_SERVICE"
CHALLENGER_SERVICE_NAME = "FRAUD_RF_SERVICE"

# Compute pool and service scaling.
# SYSTEM_COMPUTE_POOL_CPU is a shared pool; set a custom name to create one.
COMPUTE_POOL = "SYSTEM_COMPUTE_POOL_CPU"
MAX_INSTANCES = 1
INGRESS_ENABLED = True

# Gateway and traffic splitting weights.
# Adjust these values and re-run the gateway cell to update traffic distribution.
GATEWAY_NAME = "FRAUD_MODEL_GATEWAY"
CHAMPION_WEIGHT = 90
CHALLENGER_WEIGHT = 10

# Training parameters.
# Random seed for reproducibility and train/test split ratio.
SEED = 42
TEST_SIZE = 0.2

# Inference testing configuration.
# Number of sample rows to send through the gateway and the model method to call.
NUM_INFERENCE_REQUESTS = 100
INFERENCE_METHOD = "predict"

# Cleanup flags.
# Set RUN_CLEANUP to True to tear down all resources when done.
# DROP_COMPUTE_POOL controls whether the compute pool is also removed.
DROP_COMPUTE_POOL = False
RUN_CLEANUP = False

## Required Packages

In [ ]:
import os
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from snowflake.ml.model import task

In [ ]:
!pip install snowflake-ml-python==1.25.0 --quiet

## Connection Setup

In [ ]:
def get_login_token():
    with open('/snowflake/session/token', 'r') as f:
        return f.read()

connection_params = {
    "account": os.getenv('SNOWFLAKE_ACCOUNT'),
    "host": os.getenv('SNOWFLAKE_HOST'),
    "authenticator": "oauth",
    "token": get_login_token(),
    "database": SF_DATABASE,
    "schema": SF_SCHEMA,
}
if SF_ROLE:
    connection_params["role"] = SF_ROLE
if SF_WAREHOUSE:
    connection_params["warehouse"] = SF_WAREHOUSE

session = Session.builder.configs(connection_params).create()

print(f"Connected to Snowflake: {session.get_current_account()}")
print(f"  Role:      {session.get_current_role()}")
print(f"  Warehouse: {session.get_current_warehouse()}")
print(f"  Database:  {session.get_current_database()}")
print(f"  Schema:    {session.get_current_schema()}")

## Verify Privileges

In [ ]:
grants = session.sql(f"SHOW GRANTS TO ROLE {SF_ROLE}").collect()
grant_set = set()
for row in grants:
    row_dict = row.as_dict() if hasattr(row, 'as_dict') else dict(row)
    privilege = row_dict.get("privilege", "").upper()
    granted_on = row_dict.get("granted_on", "").upper()
    name = row_dict.get("name", "").upper()
    grant_set.add((privilege, granted_on, name))

required = [
    ("CREATE MODEL", "SCHEMA", f"{SF_DATABASE}.{SF_SCHEMA}",
     f"GRANT CREATE MODEL ON SCHEMA {SF_DATABASE}.{SF_SCHEMA} TO ROLE {SF_ROLE};"),
    ("CREATE SERVICE", "SCHEMA", f"{SF_DATABASE}.{SF_SCHEMA}",
     f"GRANT CREATE SERVICE ON SCHEMA {SF_DATABASE}.{SF_SCHEMA} TO ROLE {SF_ROLE};"),
    ("CREATE GATEWAY", "SCHEMA", f"{SF_DATABASE}.{SF_SCHEMA}",
     f"GRANT CREATE GATEWAY ON SCHEMA {SF_DATABASE}.{SF_SCHEMA} TO ROLE {SF_ROLE};"),
    ("BIND SERVICE ENDPOINT", "ACCOUNT", "",
     f"GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE {SF_ROLE};"),
    ("USAGE", "COMPUTE_POOL", COMPUTE_POOL,
     f"GRANT USAGE ON COMPUTE POOL {COMPUTE_POOL} TO ROLE {SF_ROLE};"),
]

missing = []
for privilege, granted_on, name, fix_sql in required:
    found = any(
        p == privilege and g == granted_on and (not name or name.upper() in n)
        for p, g, n in grant_set
    )
    if not found:
        missing.append((privilege, fix_sql))

if missing:
    print(f"WARNING: {len(missing)} required privilege(s) may be missing for role {SF_ROLE}:\n")
    for priv, fix in missing:
        print(f"  [MISSING] {priv}")
        print(f"            Fix: {fix}\n")
    print("Run the above GRANT statements as ACCOUNTADMIN before proceeding.")
else:
    print(f"All required privileges verified for role {SF_ROLE}.")

## Data Loading

In [ ]:
df = pd.read_csv(LOCAL_DATA_PATH)

print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")
df.head()

In [ ]:
DROP_COLS = ["transaction_id", "timestamp", "customer_id", "currency", "item_name"]
DISPOSABLE_DOMAINS = {
    "tempmail.com", "throwaway.email", "guerrillamail.com",
    "mailinator.com", "yopmail.com", "trashmail.net",
}
HIGH_RISK_COUNTRIES = {"NG", "RO", "PH", "UA", "ID"}
CATEGORICAL_COLS = [
    "customer_email_domain", "customer_ip_country", "billing_country",
    "shipping_country", "payment_method", "card_type", "item_category",
    "device_type", "browser",
]

df.columns = df.columns.str.lower()
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

df["is_disposable_email"] = df["customer_email_domain"].isin(DISPOSABLE_DOMAINS).astype(int)
df["is_high_risk_ip"] = df["customer_ip_country"].isin(HIGH_RISK_COUNTRIES).astype(int)
df["amount_per_item"] = (df["transaction_amount"] / df["item_quantity"]).round(2)
df["card_type"] = df["card_type"].fillna("none")

label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

y = df.pop("is_fraud")
X = df

print(f"Features: {X.shape[1]}, Samples: {X.shape[0]:,}")
print(f"Fraud: {(y == 1).sum():,}, Legit: {(y == 0).sum():,}")

## Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y,
)

n_legit = (y_train == 0).sum()
n_fraud = (y_train == 1).sum()
scale_pos_weight = n_legit / n_fraud

print(f"Train: {len(y_train):,} ({n_fraud} fraud, {n_legit} legit)")
print(f"Test:  {len(y_test):,}")
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=SEED,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_report = classification_report(y_test, xgb_pred, target_names=["Legit", "Fraud"], output_dict=True)
xgb_metrics = {
    "accuracy": float(xgb_report["accuracy"]),
    "fraud_precision": float(xgb_report["Fraud"]["precision"]),
    "fraud_recall": float(xgb_report["Fraud"]["recall"]),
    "fraud_f1": float(xgb_report["Fraud"]["f1-score"]),
    "roc_auc": float(roc_auc_score(y_test, xgb_prob)),
    "pr_auc": float(average_precision_score(y_test, xgb_prob)),
}

print(f"Champion (XGBoost) trained")
print(f"  ROC-AUC: {xgb_metrics['roc_auc']:.4f}  PR-AUC: {xgb_metrics['pr_auc']:.4f}")

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_report = classification_report(y_test, rf_pred, target_names=["Legit", "Fraud"], output_dict=True)
rf_metrics = {
    "accuracy": float(rf_report["accuracy"]),
    "fraud_precision": float(rf_report["Fraud"]["precision"]),
    "fraud_recall": float(rf_report["Fraud"]["recall"]),
    "fraud_f1": float(rf_report["Fraud"]["f1-score"]),
    "roc_auc": float(roc_auc_score(y_test, rf_prob)),
    "pr_auc": float(average_precision_score(y_test, rf_prob)),
}

print(f"Challenger (Random Forest) trained")
print(f"  ROC-AUC: {rf_metrics['roc_auc']:.4f}  PR-AUC: {rf_metrics['pr_auc']:.4f}")

In [ ]:
print("=" * 60)
print("HEAD-TO-HEAD COMPARISON")
print("=" * 60)
header = f"{'Metric':<28} {'XGBoost':>12} {'Random Forest':>14} {'Winner':>10}"
print(header)
print("-" * len(header))

comparisons = [
    ("Fraud Precision", "fraud_precision"),
    ("Fraud Recall", "fraud_recall"),
    ("Fraud F1", "fraud_f1"),
    ("Accuracy", "accuracy"),
    ("ROC-AUC", "roc_auc"),
    ("PR-AUC", "pr_auc"),
]
for label, key in comparisons:
    xv = xgb_metrics[key]
    rv = rf_metrics[key]
    winner = "XGBoost" if xv > rv else ("RF" if rv > xv else "Tie")
    print(f"  {label:<26} {xv:>11.4f} {rv:>13.4f} {winner:>10}")

## Model Registry Registration

In [ ]:
reg = Registry(session=session, database_name=SF_DATABASE, schema_name=SF_SCHEMA)
sample_input = X_test.head(100)

print(f"Registering champion: {CHAMPION_MODEL_NAME} {MODEL_VERSION}...")
try:
    xgb_mv = reg.log_model(
        xgb_model,
        model_name=CHAMPION_MODEL_NAME,
        version_name=MODEL_VERSION,
        sample_input_data=sample_input,
        conda_dependencies=["xgboost"],
        metrics=xgb_metrics,
        task=task.Task.TABULAR_BINARY_CLASSIFICATION,
        comment="Champion model - XGBoost fraud classifier (300 trees, depth 6)",
    )
    print(f"  Registered: {CHAMPION_MODEL_NAME} {MODEL_VERSION}")
except Exception as e:
    if "already exist" in str(e).lower():
        print(f"  Already exists, fetching reference...")
        xgb_mv = reg.get_model(CHAMPION_MODEL_NAME).version(MODEL_VERSION)
    else:
        raise

print(f"Registering challenger: {CHALLENGER_MODEL_NAME} {MODEL_VERSION}...")
try:
    rf_mv = reg.log_model(
        rf_model,
        model_name=CHALLENGER_MODEL_NAME,
        version_name=MODEL_VERSION,
        sample_input_data=sample_input,
        conda_dependencies=["scikit-learn"],
        metrics=rf_metrics,
        task=task.Task.TABULAR_BINARY_CLASSIFICATION,
        comment="Challenger model - Random Forest fraud classifier (500 trees, depth 12, balanced)",
    )
    print(f"  Registered: {CHALLENGER_MODEL_NAME} {MODEL_VERSION}")
except Exception as e:
    if "already exist" in str(e).lower():
        print(f"  Already exists, fetching reference...")
        rf_mv = reg.get_model(CHALLENGER_MODEL_NAME).version(MODEL_VERSION)
    else:
        raise

models_df = reg.show_models()
print("\nModels in registry:")
print(models_df[["name", "default_version_name", "comment"]].to_string(index=False))

## Model Deployment

In [ ]:
pools = [row["name"] for row in session.sql("SHOW COMPUTE POOLS").collect()]
print(f"Available compute pools: {pools}")

if COMPUTE_POOL not in pools and not COMPUTE_POOL.startswith("SYSTEM_COMPUTE_POOL"):
    session.sql(f"""
        CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL}
        MIN_NODES = 1 MAX_NODES = 1
        INSTANCE_FAMILY = CPU_X64_XS
    """).collect()
    print(f"Created compute pool: {COMPUTE_POOL}")
else:
    print(f"Using compute pool: {COMPUTE_POOL}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

model_fqn_champion = f"{SF_DATABASE}.{SF_SCHEMA}.{CHAMPION_MODEL_NAME}"
model_fqn_challenger = f"{SF_DATABASE}.{SF_SCHEMA}.{CHALLENGER_MODEL_NAME}"

def deploy_service(model_fqn, version, service_name):
    mv = reg.get_model(model_fqn).version(version)
    mv.create_service(
        service_name=service_name,
        service_compute_pool=COMPUTE_POOL,
        ingress_enabled=INGRESS_ENABLED,
        max_instances=MAX_INSTANCES,
        autocapture=True,
    )
    return service_name

print("Deploying both services in...")
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {
        executor.submit(deploy_service, model_fqn_champion, MODEL_VERSION, CHAMPION_SERVICE_NAME): "Champion",
        executor.submit(deploy_service, model_fqn_challenger, MODEL_VERSION, CHALLENGER_SERVICE_NAME): "Challenger",
    }
    for future in as_completed(futures):
        label = futures[future]
        svc = future.result()
        print(f"  {label} ({svc}) deployed.")

print("\nServices:")
for svc in [CHAMPION_SERVICE_NAME, CHALLENGER_SERVICE_NAME]:
    result = session.sql(f"DESCRIBE SERVICE {SF_DATABASE}.{SF_SCHEMA}.{svc}").collect()
    if result:
        row = result[0]
        row_dict = row.as_dict() if hasattr(row, 'as_dict') else dict(row)
        print(f"  {svc}: {row_dict.get('status', 'UNKNOWN')}")

## Gateway & Shadow / Canary Deployment

In [ ]:
gateway_fqn = f"{SF_DATABASE}.{SF_SCHEMA}.{GATEWAY_NAME}"
service1_endpoint = f"{SF_DATABASE}.{SF_SCHEMA}.{CHAMPION_SERVICE_NAME}!inference"
service2_endpoint = f"{SF_DATABASE}.{SF_SCHEMA}.{CHALLENGER_SERVICE_NAME}!inference"

spec = f"""
spec:
  type: traffic_split
  split_type: custom
  targets:
    - type: endpoint
      value: {service1_endpoint}
      weight: {CHAMPION_WEIGHT}
    - type: endpoint
      value: {service2_endpoint}
      weight: {CHALLENGER_WEIGHT}
"""

session.sql(f"CREATE OR REPLACE GATEWAY {gateway_fqn} FROM SPECIFICATION $${spec}$$").collect()
print(f"Gateway '{GATEWAY_NAME}' created with {CHAMPION_WEIGHT}/{CHALLENGER_WEIGHT} traffic split")
print(f"  Champion  ({CHAMPION_SERVICE_NAME}): {CHAMPION_WEIGHT}%")
print(f"  Challenger ({CHALLENGER_SERVICE_NAME}): {CHALLENGER_WEIGHT}%")

In [ ]:
import time as _time

MAX_WAIT_SECONDS = 300
POLL_INTERVAL = 10
elapsed = 0

print(f"Waiting for gateway '{GATEWAY_NAME}' to provision (timeout {MAX_WAIT_SECONDS}s)...")
while elapsed < MAX_WAIT_SECONDS:
    result = session.sql(f"DESC GATEWAY {gateway_fqn}").collect()
    if result:
        row_dict = result[0].as_dict() if hasattr(result[0], 'as_dict') else dict(result[0])
        ingress_url = row_dict.get('ingress_url', '')

        if 'Endpoints provisioning' not in ingress_url:
            ingress_url = f"https://{ingress_url}"
            print(f"\nGateway: {row_dict.get('name', GATEWAY_NAME)}")
            print(f"Ingress URL: {ingress_url}")
            break

    print(f"  [{elapsed:>3}s] Provisioning...", flush=True)
    _time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    print(f"\nTimeout after {MAX_WAIT_SECONDS}s. Gateway may still be provisioning — re-run this cell to check again.")

In [ ]:
updated_spec = f"""
spec:
  type: traffic_split
  split_type: custom
  targets:
    - type: endpoint
      value: {service1_endpoint}
      weight: {CHAMPION_WEIGHT}
    - type: endpoint
      value: {service2_endpoint}
      weight: {CHALLENGER_WEIGHT}
"""

session.sql(f"ALTER GATEWAY {gateway_fqn} FROM SPECIFICATION $${updated_spec}$$").collect()
print(f"Gateway '{GATEWAY_NAME}' updated to {CHAMPION_WEIGHT}/{CHALLENGER_WEIGHT} traffic split")
print(f"  Champion  ({CHAMPION_SERVICE_NAME}): {CHAMPION_WEIGHT}%")
print(f"  Challenger ({CHALLENGER_SERVICE_NAME}): {CHALLENGER_WEIGHT}%")

## Inference Testing

Send live inference requests through the gateway endpoint to verify the deployed models are serving predictions correctly.

In [ ]:
import json
import time
import os
import requests
import snowflake.connector

gateway_fqn = f"{SF_DATABASE}.{SF_SCHEMA}.{GATEWAY_NAME}"
gw_result = session.sql(f"DESC GATEWAY {gateway_fqn}").collect()
gw_dict = gw_result[0].as_dict() if hasattr(gw_result[0], 'as_dict') else dict(gw_result[0])
gw_host = gw_dict.get("ingress_url", "")
if not gw_host:
    raise RuntimeError(f"Gateway {GATEWAY_NAME} has no ingress URL yet — wait for provisioning and re-run this cell.")

gateway_url = f"https://{gw_host}/{INFERENCE_METHOD}"

PAT_NAME = "NOTEBOOK_INFERENCE_PAT"
try:
    session.sql(f"ALTER USER REMOVE PAT {PAT_NAME}").collect()
except:
    pass
pat_result = session.sql(
    f"ALTER USER ADD PAT {PAT_NAME} DAYS_TO_EXPIRY = 1 MINS_TO_BYPASS_NETWORK_POLICY_REQUIREMENT = 60 COMMENT = 'Temp token for notebook inference testing'"
).collect()
pat_row = pat_result[0].as_dict() if hasattr(pat_result[0], 'as_dict') else dict(pat_result[0])
pat_secret = pat_row.get("token_secret", pat_row.get("TOKEN_SECRET", ""))
if not pat_secret:
    print(f"PAT result keys: {list(pat_row.keys())}")
    raise RuntimeError("Could not retrieve PAT secret from ALTER USER result")
print(f"PAT created: {PAT_NAME} ({len(pat_secret)} chars)")

sf_account = os.environ.get("SNOWFLAKE_ACCOUNT")
sf_user = session.sql("SELECT CURRENT_USER()").collect()[0][0]

pat_conn = snowflake.connector.connect(
    account=sf_account,
    user=sf_user,
    authenticator="programmatic_access_token",
    token=pat_secret,
    database=SF_DATABASE,
    schema=SF_SCHEMA,
    role=SF_ROLE,
    session_parameters={"PYTHON_CONNECTOR_QUERY_RESULT_FORMAT": "json"},
)
print(f"PAT connection established (user={sf_user})")

token_data = pat_conn._rest._token_request("ISSUE")
ingress_token = token_data["data"]["sessionToken"]
print(f"Ingress token acquired ({len(ingress_token)} chars)")

headers = {
    "Authorization": f'Snowflake Token="{ingress_token}"',
    "Content-Type": "application/json",
}

print(f"\nGateway URL:   {gateway_url}")
print(f"Traffic split: Champion {CHAMPION_WEIGHT}% / Challenger {CHALLENGER_WEIGHT}%")
print(f"Requests:      {NUM_INFERENCE_REQUESTS}  |  Method: {INFERENCE_METHOD}")

indices = np.random.choice(len(X_test), size=NUM_INFERENCE_REQUESTS, replace=NUM_INFERENCE_REQUESTS > len(X_test))
X_sample = X_test.iloc[indices].reset_index(drop=True)
y_sample = y_test.iloc[indices].reset_index(drop=True)

print(f"\nSending {NUM_INFERENCE_REQUESTS} HTTP request(s) to gateway...")
print("-" * 70)

success = 0
correct = 0
total_ms = 0.0

for i in range(NUM_INFERENCE_REQUESTS):
    actual_label = int(y_sample.iloc[i])
    split_obj = json.loads(X_sample.iloc[[i]].to_json(orient="split"))
    payload = {"dataframe_split": split_obj}

    elapsed_ms = 0.0
    max_retries = 4
    status = None
    body = ""

    for attempt in range(1, max_retries + 1):
        try:
            t0 = time.perf_counter()
            resp = requests.post(gateway_url, headers=headers, json=payload, timeout=30)
            elapsed_ms = (time.perf_counter() - t0) * 1000
            status = resp.status_code
            body = resp.text[:500]
            if status == 200:
                success += 1
                break
            if attempt == 1:
                print(f"  [{i+1}/{NUM_INFERENCE_REQUESTS}]  HTTP {status} (attempt {attempt}/{max_retries})")
                print(f"         Response: {body[:300]}")
            if attempt < max_retries:
                time.sleep(10)
            else:
                print(f"  [{i+1}/{NUM_INFERENCE_REQUESTS}]  Gave up after {max_retries} attempts (HTTP {status})")
        except requests.RequestException as e:
            elapsed_ms = (time.perf_counter() - t0) * 1000
            status = "ERR"
            body = str(e)[:500]
            if attempt == 1:
                print(f"  [{i+1}/{NUM_INFERENCE_REQUESTS}]  Request error: {body[:200]}")
            if attempt < max_retries:
                time.sleep(10)
            else:
                print(f"  [{i+1}/{NUM_INFERENCE_REQUESTS}]  Gave up after {max_retries} attempts")
                break
    total_ms += elapsed_ms

    actual_tag = "FRAUD" if actual_label == 1 else "LEGIT"
    prediction_str = ""
    try:
        resp_json = json.loads(body)
        if "data" in resp_json and resp_json["data"]:
            row_data = resp_json["data"][0]
            pred_dict = row_data[1] if len(row_data) > 1 else row_data[0]
            if isinstance(pred_dict, dict):
                pred_val = pred_dict.get("output_feature_0", pred_dict)
                pred_tag = "FRAUD" if pred_val == 1 else "LEGIT"
                prediction_str = f"Predicted: {pred_tag}"
                if pred_val == actual_label:
                    correct += 1
            else:
                prediction_str = f"Raw: {pred_dict}"
    except (json.JSONDecodeError, IndexError, TypeError):
        prediction_str = f"Raw: {body[:200]}"

    print(f"  [{i+1}/{NUM_INFERENCE_REQUESTS}]  Status: {status}  |  {elapsed_ms:>7.0f}ms  |  Actual: {actual_tag}  |  {prediction_str}")

pat_conn.close()
try:
    session.sql(f"ALTER USER REMOVE PAT {PAT_NAME}").collect()
    print(f"\nPAT {PAT_NAME} removed.")
except Exception as e:
    print(f"\nWarning: Could not remove PAT: {e}")

avg_ms = total_ms / NUM_INFERENCE_REQUESTS if NUM_INFERENCE_REQUESTS else 0
print("\n" + "=" * 70)
print(f"SUMMARY: {success}/{NUM_INFERENCE_REQUESTS} requests returned HTTP 200")
if success > 0:
    print(f"         {correct}/{success} predictions matched actual label")
print(f"         Avg response time: {avg_ms:.0f}ms  |  Total: {total_ms:.0f}ms")
print(f"         Gateway: {gateway_url}")
print("=" * 70)

## Model Serving Metrics

Query the autocaptured inference tables for both models to see request counts, success rates, and average latency over the last 60 minutes.

In [ ]:
metrics_sql = f"""
SELECT
    '{CHAMPION_MODEL_NAME}' AS MODEL_NAME,
    'Champion' as "A/B Test",
    COUNT(*) AS TOTAL_REQUESTS,
    SUM(CASE WHEN record_attributes:"snow.model_serving.response.code"::VARCHAR LIKE '2%' THEN 1 ELSE 0 END) AS SUCCESS_COUNT,
FROM TABLE(INFERENCE_TABLE('{SF_DATABASE}.{SF_SCHEMA}.{CHAMPION_MODEL_NAME}'))
WHERE record_attributes:"snow.model_serving.request.timestamp" >= DATEADD(minute, -60, CURRENT_TIMESTAMP())

UNION ALL

SELECT
    '{CHALLENGER_MODEL_NAME}' AS MODEL_NAME,
    'Challenger' as "A/B Test",
    COUNT(*) AS TOTAL_REQUESTS,
    SUM(CASE WHEN record_attributes:"snow.model_serving.response.code"::VARCHAR LIKE '2%' THEN 1 ELSE 0 END) AS SUCCESS_COUNT
FROM TABLE(INFERENCE_TABLE('{SF_DATABASE}.{SF_SCHEMA}.{CHALLENGER_MODEL_NAME}'))
WHERE record_attributes:"snow.model_serving.request.timestamp" >= DATEADD(minute, -60, CURRENT_TIMESTAMP())
"""

metrics_df = session.sql(metrics_sql).to_pandas()
print("Model Serving Metrics (last 60 minutes)")
print("=" * 70)
print(metrics_df.to_string(index=False))
print("=" * 70)

## Things to Try

Now that both models are deployed and serving traffic through the gateway, here are some experiments you can run. After making changes, re-run the **Inference Testing** cell (and the metrics cell above) to observe the impact.

**Tip:** Increase `NUM_INFERENCE_REQUESTS` to 500 or 1000 for more statistically meaningful results.

### Ideas

1. **Change the traffic split** — Update the weights below and run the next cell to shift more (or all) traffic to the challenger model.
2. **Register and deploy a V2 model** — Retrain with different hyperparameters, register as `MODEL_VERSION = "V2"`, deploy a new service, and add it to the gateway.
3. **Promote the challenger** — Set the champion weight to 0 and challenger to 100 to fully cut over.
4. **Drop a model from the gateway** — Remove one of the targets from the gateway spec to serve traffic from a single model.
5. **Compare metrics over time** — Re-run the metrics cell periodically to watch latency and success rates change as you adjust the deployment.

In [ ]:
NEW_CHALLENGER_WEIGHT = 100
NEW_NUM_INFERENCE_REQUESTS = 500
NEW_CHAMPION_WEIGHT = 100 - NEW_CHALLENGER_WEIGHT

gateway_fqn = f"{SF_DATABASE}.{SF_SCHEMA}.{GATEWAY_NAME}"
service1_endpoint = f"{SF_DATABASE}.{SF_SCHEMA}.{CHAMPION_SERVICE_NAME}!inference"
service2_endpoint = f"{SF_DATABASE}.{SF_SCHEMA}.{CHALLENGER_SERVICE_NAME}!inference"

new_spec = f"""
spec:
  type: traffic_split
  split_type: custom
  targets:
    - type: endpoint
      value: {service1_endpoint}
      weight: {NEW_CHAMPION_WEIGHT}
    - type: endpoint
      value: {service2_endpoint}
      weight: {NEW_CHALLENGER_WEIGHT}
"""

session.sql(f"ALTER GATEWAY {gateway_fqn} FROM SPECIFICATION $${new_spec}$$").collect()
print(f"Gateway updated: Champion {NEW_CHAMPION_WEIGHT}% / Challenger {NEW_CHALLENGER_WEIGHT}%")

CHAMPION_WEIGHT = NEW_CHAMPION_WEIGHT
CHALLENGER_WEIGHT = NEW_CHALLENGER_WEIGHT
NUM_INFERENCE_REQUESTS = NEW_NUM_INFERENCE_REQUESTS
print(f"NUM_INFERENCE_REQUESTS updated to {NUM_INFERENCE_REQUESTS}")
print("\nRe-run the Inference Testing cell and Metrics cell to see the results.")

## Cleanup (Optional)

Drop services, gateway, and models created by this notebook. Set `DROP_COMPUTE_POOL` to `True` in the configuration cell to also remove the compute pool.

In [ ]:
if not RUN_CLEANUP:
    print("Cleanup skipped. Set RUN_CLEANUP = True in the configuration cell to enable.")
else:
    fqn = lambda name: f"{SF_DATABASE}.{SF_SCHEMA}.{name}"

    cleanup_steps = [
        ("Service", f"DROP SERVICE IF EXISTS {fqn(CHAMPION_SERVICE_NAME)}"),
        ("Service", f"DROP SERVICE IF EXISTS {fqn(CHALLENGER_SERVICE_NAME)}"),
        ("Gateway", f"DROP GATEWAY IF EXISTS {fqn(GATEWAY_NAME)}"),
        ("Model", f"DROP MODEL IF EXISTS {fqn(CHAMPION_MODEL_NAME)}"),
        ("Model", f"DROP MODEL IF EXISTS {fqn(CHALLENGER_MODEL_NAME)}"),
    ]

    if DROP_COMPUTE_POOL and not COMPUTE_POOL.startswith("SYSTEM_COMPUTE_POOL"):
        cleanup_steps.append(("Compute Pool", f"DROP COMPUTE POOL IF EXISTS {COMPUTE_POOL}"))

    rows = session.sql(f"SHOW SERVICES IN SCHEMA {SF_DATABASE}.{SF_SCHEMA}").collect()
    for row in rows:
        rd = row.as_dict() if hasattr(row, 'as_dict') else dict(row)
        if rd.get("is_job") and rd.get("managing_object_name", "") in [
            f"{SF_DATABASE}.{SF_SCHEMA}.{CHAMPION_MODEL_NAME}",
            f"{SF_DATABASE}.{SF_SCHEMA}.{CHALLENGER_MODEL_NAME}",
        ]:
            cleanup_steps.append(("Build Job", f"DROP SERVICE IF EXISTS {fqn(rd['name'])}"))

    print("Running cleanup...")
    for label, sql in cleanup_steps:
        try:
            session.sql(sql).collect()
            print(f"  [OK]   {label}: {sql}")
        except Exception as e:
            print(f"  [FAIL] {label}: {sql}\n         {e}")

    print("\nCleanup complete.")